## Lab 12: CLIP, DiNO, DETR, SAM, and Gemma 4
*Suggested time: 55-60 minutes*


We show demonstrate the following models,

-	Multi-modal (vision-text) CLIP (Contrastive Language-Image Pre-training) Model
-	Detection Transformer (DETR) based Object detection model
-	Segment Anything (SAM) model
-	Automatic Speech Recognition (ASR) model
-	Multi-modal (vision, audio, text) Gemma 4 model for edge devices



**A. CLIP(Contrastive Language-Image Pre-training)Model**

Step 0:
- Install needed packages and upload sample images

In [ ]:
!pip install -q transformers torch

In [ ]:
!wget https://edge-ai-doulos.s3.us-west-2.amazonaws.com/images.zip
!unzip -q images.zip

**Step 1:**
- Perform zero-shot image classification
- Use Open AI clip-vit-base-patch32 CLIP model
- Provide three labels and use a picture matching one of the labels
- Check if the model correctly maps the image with the label

In [ ]:
from PIL import Image
import requests

from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

image_location = 'images/two_dogs.png'
image = Image.open(image_location)

inputs = processor(text=["picture of elephant", "a photo of a cat", "a photo of a dog"], images=image, return_tensors="pt", padding=True)

outputs = model(**inputs)
logits_per_image = outputs.logits_per_image # this is the image-text similarity score
probs = logits_per_image.softmax(dim=1) # we can take the softmax to get the label probabilities
print(f"Probabilities: {probs}")

**Step 2:** Self-supervised learning model - DINO (DIstillation with NO Labels)

- Use a self-learning model to extract features
- Model (DINO) based on Vision Transformer (ViT) architecture



In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
import requests
import matplotlib.pyplot as plt
import numpy as np

# Load DINO ViT-Small (8x8 patches for higher resolution detail)
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model = torch.hub.load('facebookresearch/dino:main', 'dino_vits8').to(device)
model.eval()

# Preprocessing: DINO expects 224x224 or multiples of the patch size (8)
transform = T.Compose([
    T.Resize(480), # High res for better visualization
    T.CenterCrop(480),
    T.ToTensor(),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

# Load an image
image_location = 'images/macaw_2.png'
img_raw = Image.open(image_location).convert('RGB')
img_tensor = transform(img_raw).unsqueeze(0).to(device)

- The different DINO attention heads visualize distinct semantic regions or features learned by the model without explicit supervision.
- In self-supervised Vision Transformers, each attention head can specialize in a different way of relating image patches to the `[CLS]` token.
- Common emergent behaviors include:
  - **Object segmentation:** some heads strongly attend to the foreground object, separating it from the background.
  - **Background attention:** other heads focus on contextual background regions.
  - **Feature detection:** some heads respond to textures, shapes, or object parts regardless of exact position.
  - **Diversity of focus:** each head produces a unique attention pattern, giving the model richer visual representation.
- In the plotted subplots, each head shows one attention map from DINO.
- The varying intensity patterns across heads suggest unsupervised object discovery or segmentation-like behavior.


In [ ]:
# 1. Forward pass to get attention
# We use 'get_last_selfattention' which is a built-in method for the DINO hub model
with torch.no_grad():
    attentions = model.get_last_selfattention(img_tensor)

# 2. Process the attention map
# Structure: [Batch, Heads, Num_Patches+1, Num_Patches+1]
nh = attentions.shape[1] # Number of attention heads (usually 6 for ViT-S)
w_featmap = img_tensor.shape[-2] // 8
h_featmap = img_tensor.shape[-1] // 8

# We keep only the attention of the [CLS] token to the other patches
# Index 0 is the CLS token; we take 0, :, 0, 1:
attentions = attentions[0, :, 0, 1:].reshape(nh, w_featmap, h_featmap)

# 3. Visualize the Different "Heads"
fig, axs = plt.subplots(1, nh + 1, figsize=(20, 5))
axs[0].imshow(img_raw.resize((480, 480)))
axs[0].set_title("Original")
axs[0].axis('off')

for i in range(nh):
    axs[i+1].imshow(attentions[i].cpu().numpy(), cmap='magma')
    axs[i+1].set_title(f"Head {i}")
    axs[i+1].axis('off')

plt.show()

The images showing **'Head 0' to 'Head 5'** are visualizations of the **attention heads from the last Transformer block** of the DINO model.

- The notebook uses `model.get_last_selfattention(img_tensor)` to extract the attention maps.
- This method returns the **self-attention weights from the final layer** of the Vision Transformer encoder.
- The last-layer attention maps are often the most **semantically meaningful**, so they are useful for interpreting what the model focuses on.
- Each head (`Head 0` to `Head 5`) represents a different attention pattern learned by the model.

- The number of attention heads (nh) in a DINO model can be determined from the shape of the extracted attention tensor.
- After a forward pass, attentions has shape [Batch, Heads, Num_Patches+1, Num_Patches+1].
- The number of heads is obtained with nh = attentions.shape[1].
- In the example using dino_vits8, the model has attention heads.


**Step 3:** Use of DINO attention heads

- As seen in the earlier code, attention heads can be used for segmentation
- In this code we use PCA to convert the features from attention heads to a 2D embedding

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from PIL import Image
import requests
import torchvision.transforms as T

# 1. Load Model (DINOv2)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
model.eval()

# 2. Load and Preprocess Image
image_location = 'images/macaw_2.png'
img = Image.open(image_location).convert('RGB')

w, h = img.size
# Resize to a multiple of patch size (14)
img_t = T.Compose([
    T.Resize((448, 448)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])(img).unsqueeze(0).to(device)

# 3. Extract Patch Features
with torch.no_grad():
    features_dict = model.forward_features(img_t)
    features = features_dict['x_norm_patchtokens'] # Shape: [1, 1024, 384]

# 4. Perform PCA (Reduce 384-D -> 3-D)
# Flatten to [Num_Patches, Embedding_Dim]
features = features.squeeze(0).cpu().numpy()
pca = PCA(n_components=3)
pca_features = pca.fit_transform(features)

# 5. Normalize and Reshape to RGB Image
# Scale features to 0-1 range for RGB visualization
pca_features = (pca_features - pca_features.min()) / (pca_features.max() - pca_features.min())

# Reshape back to grid (448/14 = 32 patches)
grid_size = 448 // 14
pca_img = pca_features.reshape(grid_size, grid_size, 3)

# 6. Plot Results
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img.resize((448, 448)))
plt.title("Original Image")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(pca_img)
plt.title("DINO PCA Segmentation")
plt.axis('off')
plt.show()

**Step 4:**  Use of DEtection TRansfomer (DETR) model

- Download RF-DETR model
- Work with Nano and Base version of model
- RF-DETR uses DINO for pre-processing

In [ ]:
!pip uninstall -q transformers -y
!pip install -q transformers==5.6.2

In [ ]:
!pip install -q supervision

In [ ]:
!pip install -q rfdetr

In [ ]:
import requests
import supervision as sv
from PIL import Image
from rfdetr import RFDETRNano
from rfdetr.assets.coco_classes import COCO_CLASSES

model = RFDETRNano()

image_path = "images/clock.jpg"
image = Image.open(image_path).convert("RGB")

detections = model.predict(image, threshold=0.5)

labels = [f"{COCO_CLASSES[class_id]}" for class_id in detections.class_id]

annotated_image = sv.BoxAnnotator().annotate(image, detections)
annotated_image = sv.LabelAnnotator().annotate(annotated_image, detections, labels)

sv.plot_image(annotated_image)

In [ ]:
import cv2
import requests
import supervision as sv
from PIL import Image
from io import BytesIO
from rfdetr import RFDETRBase

# 1. Load the pre-trained Base model (trained on COCO)
model = RFDETRBase()

# 2. Load an image from a URL
image_path = "images/pizza.png"
image = Image.open(image_path).convert("RGB") # Ensure image is in RGB format

# 3. Predict
# Returns a supervision Detections object
detections = model.predict(image, threshold=0.5)

# 4. Annotate and Plot
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

annotated_image = box_annotator.annotate(scene=image.copy(), detections=detections)
annotated_image = label_annotator.annotate(scene=annotated_image, detections=detections)

sv.plot_image(annotated_image)

**Step 5**: Segment Anything (SAM) Model

- Trained on the expansive SA-1B dataset, which contains more than 1 billion masks spread over 11 million carefully curated images, SAM has displayed impressive zero-shot performance, surpassing previous fully supervised results in many cases.
- Install and work with SAM models provided by Ultralytics


In [ ]:
!pip install -q ultralytics

In [ ]:
import ultralytics
ultralytics.checks()

## Segment Anything

You can [segment](https://docs.ultralytics.com/tasks/segment/) specific objects in an image or video using different prompts, such as bounding box and point prompts.

### Bounding box prompt

The `bbox_prompt` refers to a bounding box input that guides the model in segmenting a specific object within an image. In the example below, you will segment only the bus by providing the bounding box coordinates.

In [ ]:
from ultralytics import SAM

# Load a model
model = SAM("mobile_sam.pt")

# Run inference with bboxes prompt (Provide the bounding box coordinates
# for the path, ensuring that only path is segmented in the entire image)
results = model("images/school_bus.png",
                bboxes=[3.8328723907470703, 229.35601806640625,
                        796.2098999023438, 728.4313354492188])

results[0].show()  # Display results

### Point prompt

The `point_prompt` refers to a specific point input (x, y) that guides the Segment Anything Model (SAM) in segmenting an object within an image. Instead of providing a bounding box, you can indicate an object by selecting a point on it, and SAM will generate a segmentation mask around that point.

In [ ]:
from ultralytics import SAM

# Load a model
model = SAM("mobile_sam.pt")

# Run inference with point prompt (Provide the point coordinates for the
# table area, ensuring that only the table is segmented in the entire image)
results = model("images/pizza.png",
                points=[34, 714])

results[0].show()  # Display results

### Multiple points prompt

Multiple `point_prompt` inputs refer to specific points (x, y) that serve as prompts to guide the Segment Anything Model (SAM) in segmenting multiple objects within an image.

In [ ]:
from ultralytics import SAM

# Load a model
model = SAM("mobile_sam.pt")

# Run inference with multiple point prompts (Provide the points coordinates for
# path area, ensuring that only the path is segmented in the entire image)
results = model("images/car_downtown.png",
                points=[[34, 714], [283, 634]])

results[0].show()  # Display results

**Step 6:**  Moonshine ASR Model

- Install Moonshine model
- Try out transcription of pre-recorded model using example from Moonshine documentation


In [ ]:
!pip install moonshine-voice

## Transcribing a File

Moonshine Voice is focused on applications where you need to understand user's speech in real time, but since notebooks like this don't allow microphone access, the first example we'll show will run on [a pre-recorded audio file](https://github.com/moonshine-ai/moonshine-v2/raw/refs/heads/main/test-assets/beckett.wav) of one of my favorite Beckett quotes:

In [ ]:
# Download the audio file.
! curl -O -L 'https://github.com/moonshine-ai/moonshine/raw/refs/heads/main/test-assets/beckett.wav'

# Import the pip package for Moonshine Voice (notice the underscore, not dash!).
import moonshine_voice

# This returns a path and model type for a particular language, downloading the
# Moonshine model if necessary and caching it locally.
model_path, model_arch = moonshine_voice.get_model_for_language("en")

# Create a Transcriber object using the model information.
transcriber = moonshine_voice.Transcriber(model_path=model_path, model_arch=model_arch)

# To make testing easier, Moonshine Voice includes this convenience function to
# load the PCM audio data from a .wav file, without requiring any additional
# packages like librosa.
audio_data, sample_rate = moonshine_voice.load_wav_file("beckett.wav")

# Extract the text from the spoken audio in the file.
transcript = transcriber.transcribe_without_streaming(audio_data)

# Display the results, line by line.
for line in transcript.lines:
  print(line)

Audio("beckett.wav")